In [ ]:
%pip install pandas matplotlib seaborn sqlalchemy pymysql scikit-learn request

In [ ]:
import requests

url = 'https://api.data.go.kr/openapi/tn_pubr_public_sub_air_qur_api'
params ={'serviceKey' : '437e6185e12c66f37d4232d6e90f5b5ddad7865dee38498aee739bd85ec03351', 'pageNo' : '0', 'numOfRows' : '100', 'type' : 'xml', 'CTPV_NM' : '', 'SGG_NM' : '', 'MEASURE_YR' : '', 'SUBWAY_NO_NM' : '', 'SUBWAY_STA_NM' : '', 'MEASURE_POSI' : '', 'PM10' : '', 'PM2·5' : '', 'CO2' : '', 'CH2O' : '', 'CO' : '', 'NO2' : '', 'RN' : '', 'VOC' : '', 'MNG_INST_NM' : '', 'CRTR_YMD' : '', 'instt_code' : '', 'instt_nm' : '' }

response = requests.get(url, params=params)
print(response.content)

In [15]:
import requests # HTTP 요청 라이브러리 (API 호출)
import pandas as pd # 데이터프레임 처리 라이브러리
import xml.etree.ElementTree as Ex # XML 파싱 라이브러리
from sqlalchemy import create_engine # DB 연결 엔진 생성
from urllib.parse import quote_plus # URL 특수문자 인코딩 (비밀번호 안전 처리)
import re # 정규표현식 (컬럼명 특수문자 제거)
import sqlalchemy # SQL 실행용

In [16]:
# 컬럼 매핑 딕셔너리 - API 응답의 영문 태그명 -> MySQL에 저장할 한글 컬럼명
COLUMN_MAP = {
    'ctpvNm'     : '시도명',
    'sggNm'      : '시군구명',
    'subwayNoNm' : '호선명',
    'subwayStaNm': '역명',
    'pm10'       : '미세먼지',
    'pm2_5'      : '초미세먼지',
    'co2'        : '이산화탄소',
    'ch2o'       : '폼알데하이드',
    'co'         : '일산화탄소',
}

In [ ]:
# 공공데이터 API에서 전체 데이터 수집
url = 'https://api.data.go.kr/openapi/tn_pubr_public_sub_air_qur_api'

all_rows = [] # 전체 페이지의 수집 결과를 담을 빈 리스트
page = 1

while True:
    # API 요청 파라미터 설정
    params = {
        'serviceKey' : '437e6185e12c66f37d4232d6e90f5b5ddad7865dee38498aee739bd85ec03351',
        'pageNo' : str(page),
        'numOfRows' : '100',
        'type' : 'xml' # 응답 형식
    }
    
    response = requests.get(url, params=params) # GET 방식으로 API 호출